# EBSD to Abaqus: smoothed grain structure, conformal mesh, per-grain Bunge-Euler materials

Pipeline: read a real EBSD `.ctf` map (`EBSDReader`), crop to a demo-sized region of
interest, smooth the pixel grain boundaries into polygons (Technique A --
`polygonised_grain_structure.pix_to_geom` + `smooth_gbsegs`, pixel-exact outlines with
shared grain-boundary segments smoothed; see `confMesh2d_mcgs.ipynb` and
`confMesh2d_compare_techniques.ipynb`), mesh conformally (`mesh_gs`), then export to an
Abaqus `.inp` with one `*Material` per grain carrying its grain-averaged Bunge-Euler angles
(`phi1`, `Phi`, `phi2`, degrees).

All 2D. Uses the existing DefDAP-backed `EBSDReader` (`interfaces/defdap/ebsd_reader.py`) --
not the in-development quaternion-based `ebsd/` module (`src/upxo/ebsd/admin/EBSD_MODULE_PLAN.md`),
which is a separate, later track.

Requires `gmsh` (`pip install upxo[mesh]`) and `defdap`.

## 1. Load the EBSD map and detect grains

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from upxo.interfaces.defdap.ebsd_reader import EBSDReader

# Change to the path of your CTF file
CTF_FILE = r'C:\Development\EBSD datasets\UKAEA__OFHCCu\OFHC_Cu_dataset\EBSD_pre\warp_out_s2.ctf'

rdr = EBSDReader.from_file(CTF_FILE, min_grain_size=10, misori_tol=8)
print(rdr)

## 2. Crop to a demo-sized region of interest

The full map has ~1,500 grains with real, noisy pixel-derived boundaries. Meshing all of
them conformally at once hit a Gmsh edge-recovery failure (`Unable to recover the edge ...`)
on a sliver feature -- an expected risk with real (non-synthetic) EBSD boundaries at scale,
not specific to this dataset. A cropped region keeps the demo fast, inspectable, and within
a grain count Gmsh handles cleanly; the same crop-then-mesh approach applies to any
sub-region of the full map for a real job.

`region` is `[xstart%, ystart%, xend%, yend%]` of the full map.

In [ ]:
REGION = [35, 35, 65, 65]   # central ~20% x 30% of the map
rdr = rdr.crop(REGION, inplace=False)
print(rdr)

## 3. Characterise: fill unindexed/boundary pixels, relabel contiguous

`min_grain_size=0` here -- DefDAP already dropped sub-10-pixel grains at detection time on
the *full* map (`min_grain_size=10` above). Re-applying a `min_grain_size` filter after
cropping is unsafe: a grain truncated to a sliver by the crop boundary would be dropped from
`prop` but its pixels are not currently re-filled by `characterise()`, leaving stray
`0`-labelled pixels in `lfi_ebsd`. `min_grain_size=0` avoids that path entirely.

In [ ]:
result = rdr.characterise(connectivity=4, min_grain_size=0)
print(f"grains: {len(result['prop'])}")
print(f"lfi_ebsd: unique={np.unique(rdr.lfi_ebsd).size}, min={rdr.lfi_ebsd.min()}, max={rdr.lfi_ebsd.max()}")

### Split disconnected same-id grains

A grain is one connected pixel region by construction. Cropping to a rectangular ROI can
slice a single grain's irregular shape into two or more spatially disjoint pieces that still
share one id -- a labelling artifact, not a real grain. Left uncorrected, this breaks
pixel-exact geometrification (Technique A assumes one polygon per grain id) and silently
mis-groups unrelated pixels into one "grain" for orientation averaging. Fix it before
anything downstream sees the label field: the largest piece keeps the original id, every
other piece gets a fresh id. Per-pixel orientations are untouched, so each split-off fragment
still averages correctly from its own pixels.

In [ ]:
split_map = rdr.split_disconnected_grains(connectivity=4)
print(f"grains split: {len(split_map)}")
if split_map:
    print(f"  new_id -> original_id (first 10): "
          f"{dict(list(split_map.items())[:10])}")
print(f"grains after split: {np.unique(rdr.lfi_ebsd).size}")

In [ ]:
fig, axes = rdr.plot_grain_structure_with_boundaries(
    show_euler_panel=True, euler_channel=0,
    boundary_color=(0, 0, 0), figsize=(13, 6), dpi=120)

## 4. Grain-averaged Bunge Euler angles

Mean quaternion per grain (renormalised, positive-hemisphere), converted to Bunge Euler
(degrees) -- an arithmetic quaternion mean, not a symmetry-aware SO(3) mean; adequate for the
low intra-grain misorientation spread typical of a single EBSD grain.

In [ ]:
euler_by_gid = rdr.grain_average_euler_deg()
gid0 = next(iter(euler_by_gid))
print(f"grains with orientation: {len(euler_by_gid)}")
print(f"example grain {gid0}: phi1, Phi, phi2 (deg) = {euler_by_gid[gid0]}")

## 5. Smooth the grain boundaries into polygons (Technique A)

`polygonised_grain_structure.pix_to_geom` builds every grain outline directly from the label
image (pixel-exact), splits the boundaries at junction points, and assembles rings from
segments shared by neighbouring grains -- islands are handled by geometrifying the
hole-filled structure and each island cluster separately, then cutting each island out of its
host. `smooth_gbsegs` then smooths the shared segments (each boundary moves once, consistently
for both grains it borders).

In [ ]:
from upxo.pxtal.geometrification import polygonised_grain_structure

lfi = rdr.lfi_ebsd
geo = polygonised_grain_structure(lfi, np.unique(lfi), None)
geo.pix_to_geom(verbose=False)
geo.smooth_gbsegs(geo.GB, npasses=2, max_smooth_levels=np.repeat(2, 2),
                  plot=False, name='s')
cells = geo.smoothed['s']['GRAINS']
print(f"smoothed grain polygons: {len(cells)}")

In [ ]:
from upxo.viz import gsviz
from shapely.geometry import MultiPolygon

def polys_of(geom):
    if hasattr(geom, 'geoms'):
        return [p for g in geom.geoms for p in polys_of(g)]
    return [geom]

fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
gsviz.plot_multipolygon_geometric(
    MultiPolygon([p for g in cells.values() for p in polys_of(g)]), fig=fig, ax=ax)
ax.set_title(f'Smoothed grain structure ({len(cells)} grains)')

## 6. Conformal mesh

In [ ]:
from upxo.meshing.gsmesh2d import mesh_gs, visualize_gs_mesh

res = mesh_gs(cells, mesh_size_gb=1.0, mesh_size_bulk=12.0,
              mesh_algo=6, recombine_to_quads=False, verbose=False)
m = res['mesher']
m.form_elsets_gmsh()
print(f"nodes: {res['n_nodes']}  |  triangles: {res['n_tri']}")

In [ ]:
fig, ax = visualize_gs_mesh(res, figsize=(15, 15), dpi=250, show_nsets=False)

## 7. Export to Abaqus: physical units + per-grain Bunge-Euler materials

The mesh so far lives in pixel-index coordinates (the same space `lfi`/seeds/polygons use).
Scale node coordinates by `rdr.step_size` (microns/pixel) before export -- a uniform linear
scale applied to the already-generated mesh, safe post-hoc.

`mesh_gs` may split an irregular grain's smoothed polygon into more than one flat cell
(`res['gid_map']`: `{flat_id: original_grain_id}`); `grain_euler_deg` is remapped from
original grain id to flat id so every flat elset resolves to its parent grain's orientation.

`material_format='bunge_euler'` writes one `*Material` per grain: a comment with
`phi1/Phi/phi2` (deg) plus `*User Material, constants=3` (the same three angles) and
`*Depvar` -- ready for a crystal-plasticity UMAT to consume. This mirrors the convention
already used by the 3D exporter (`pxtal.twinned_simple_3d.abaqus_exporter_3d`).

In [ ]:
from pathlib import Path

m.nodes[:, :2] *= rdr.step_size   # pixel-index -> microns

gid_map = res['gid_map']          # {flat_id: original_grain_id}
grain_euler_deg_flat = {
    flat_id: euler_by_gid[gid]
    for flat_id, gid in gid_map.items() if gid in euler_by_gid
}

OUT_INP = Path.cwd() / 'ebsd_to_abaqus_2d_demo.inp'
m.export_abaqus_inp(
    OUT_INP, plane='stress', thickness=1.0,
    material_format='bunge_euler', grain_euler_deg=grain_euler_deg_flat,
    heading='UPXO EBSD -> smoothed mesh -> Abaqus, grain-averaged Bunge Euler materials')
print(f"written: {OUT_INP}")

In [ ]:
from upxo.meshing.writer_ABQ import summarize_inp

print(summarize_inp(OUT_INP))
text = OUT_INP.read_text(encoding='utf-8')
print('materials  :', text.count('*Material,'))
print('user mats  :', text.count('*User Material'))
print('sections   :', text.count('*Solid Section'))

i = text.index('*Material,')
print()
print(text[i:i + 220])

## 8. Element quality statistics

`report_quality()` and `report_fidelity()` already ran automatically at the end of meshing
(inside `femesh_gmsh`), so `m.quality_report`, `m.validation_report` and `m.fidelity_report`
are populated -- no extra meshing pass needed here.

* **Quality** -- aspect ratio (max/min corner-edge length) and minimum interior angle, per
  element type. `min`/`max` are the worst/best single elements; `p95` is the 95th-percentile
  aspect ratio (a well-conditioned tail) -- for min angle, the *low* end (`min`, not `p95`)
  is the one to watch, since a small minimum angle is the sign of a sliver element.
* **Validation** -- inverted (clockwise) and degenerate (zero-area) elements, and any grain
  that ended up with no elements at all.
* **Fidelity** -- how closely the mesh's per-grain area and grain-boundary length match the
  smoothed input polygons from Technique A (Section 5), the largest relative error across all
  grains.

In [ ]:
import pandas as pd

qr, vr, fr = m.quality_report, m.validation_report, m.fidelity_report

quality_table = pd.DataFrame({
    eltype: {
        'n elements': qr['aspect_ratio'][eltype]['n'],
        'aspect ratio: min': qr['aspect_ratio'][eltype]['min'],
        'aspect ratio: mean': qr['aspect_ratio'][eltype]['mean'],
        'aspect ratio: p95': qr['aspect_ratio'][eltype]['p95'],
        'aspect ratio: max': qr['aspect_ratio'][eltype]['max'],
        'min angle (deg): min': qr['min_angle_deg'][eltype]['min'],
        'min angle (deg): mean': qr['min_angle_deg'][eltype]['mean'],
        'min angle (deg): p95': qr['min_angle_deg'][eltype]['p95'],
    }
    for eltype in qr['aspect_ratio'] if qr['aspect_ratio'][eltype]['n']
})
quality_table.style.format('{:.4g}') if hasattr(quality_table, 'style') else quality_table

In [ ]:
print("Validation")
print(f"  triangles: {vr['n_tri']}  |  quads: {vr['n_quad']}")
print(f"  grains missing elements : {len(vr['grains_missing_elements'])}")
print(f"  inverted (clockwise) elements : {vr['clockwise_elements']}")
print(f"  degenerate (zero-area) elements : {vr['degenerate_elements']}")
print()
print("Fidelity vs Technique-A smoothed polygons")
print(f"  max grain-area relative error      : {fr['max_area_rel_error']:.4%}")
print(f"  max grain-GB-length relative error : {fr['max_gb_length_rel_error']:.4%}")

In [ ]:
from upxo.meshing import elemOps

ar = elemOps.compute_elementQuality_AR_2d(m.nodes, m.elConn)
ang = elemOps.compute_min_angle_deg_2d(m.nodes, m.elConn)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
for eltype in ar:
    if len(ar[eltype]) == 0:
        continue
    axes[0].hist(ar[eltype], bins=40, alpha=0.7, label=eltype)
    axes[1].hist(ang[eltype], bins=40, alpha=0.7, label=eltype)
axes[0].set_xlabel('Aspect ratio (max/min corner-edge length)')
axes[0].set_ylabel('Elements')
axes[0].set_title('Element aspect ratio')
axes[1].set_xlabel('Minimum interior angle (deg)')
axes[1].set_title('Element minimum angle')
for ax in axes:
    ax.legend()
fig.tight_layout()

In [ ]:
print(f"Minimum, mean and maximum aspect ratios: {np.round(ar[eltype].min(), 2)}, {np.round(ar[eltype].mean(), 2)} and {np.round(ar[eltype].max(), 2)}")
print(f"Minimum, mean and maximum angles: {np.round(ang[eltype].min(), 2)}, {np.round(ang[eltype].mean(), 2)} and {np.round(ang[eltype].max(), 2)}")

## 9. Interactive 3D view: FE mesh overlaid on the grain structure

PyVista, `trame` backend -- a real interactive widget (pan/zoom/rotate with the mouse), not a
static image. **Run this cell yourself** in a live kernel to see it; a widget reference baked
into a saved notebook points at a `localhost` server from that session and goes stale once it
ends, so this cell's output is intentionally left empty here.

Grain-colour fill is the mesh, coloured by owning grain (`m._elem_owners['triangle']`); the
black wireframe is the element edges; the red overlay is the smoothed Technique-A grain
boundary (`cells`, Section 5), scaled to the same microns as `m.nodes` so the two align
exactly -- zoom into any of the grains flagged in the quality-stats section (8) to see the
mesh follow the grain boundary at the pixel level.

In [ ]:
import pyvista as pv

# --- mesh: compact node numbering (mirrors export_confmesh2d_inp) ---
nodes = np.asarray(m.nodes)
valid = ~np.isnan(nodes[:, 0])
old_ids = np.where(valid)[0]
remap = np.zeros(int(old_ids.max()) + 1, dtype=int)
remap[old_ids] = np.arange(old_ids.size)
points = np.column_stack([nodes[old_ids, 0], nodes[old_ids, 1],
                          np.zeros(old_ids.size)])

conn = np.asarray(m.elConn['triangle'])[:, :3]
faces = np.hstack([np.full((len(conn), 1), 3, dtype=np.int64),
                   remap[conn]]).astype(np.int64)
mesh_pv = pv.PolyData(points, faces.ravel())
mesh_pv.cell_data['grain_id'] = m._elem_owners['triangle'].astype(float)

# --- grain-boundary overlay: Technique-A polygons, scaled to microns ---
def _ring_segments(cells_dict, scale):
    pts, lines, offset = [], [], 0
    for geom in cells_dict.values():
        polys = list(geom.geoms) if hasattr(geom, 'geoms') else [geom]
        for poly in polys:
            for ring in [poly.exterior, *poly.interiors]:
                c = np.asarray(ring.coords) * scale
                n = len(c)
                pts.append(np.column_stack([c[:, 0], c[:, 1], np.zeros(n)]))
                lines.append(np.hstack([[n], np.arange(offset, offset + n)]))
                offset += n
    return np.vstack(pts), np.concatenate(lines).astype(np.int64)

gb_points, gb_lines = _ring_segments(cells, rdr.step_size)
gb_pv = pv.PolyData(gb_points, lines=gb_lines)

pl = pv.Plotter(notebook=True, window_size=(1000, 1000))
pl.add_mesh(mesh_pv, scalars='grain_id', cmap='tab20', show_edges=True,
           edge_color='black', line_width=0.3, show_scalar_bar=False)
pl.add_mesh(gb_pv, color='red', line_width=2.0)
pl.camera_position = 'xy'
pl.enable_parallel_projection()
pl.show(jupyter_backend='trame')